# treetracker_results

`GET /datascience_results/treetracker_results/`

Per-session GPS tree-tracker results.

> **Returns 501 today.** `veritree-tree-tracker-algorithms` writes only staging/fact tables and `dim_*` rollups; it publishes no per-session results table yet. This notebook is the harness for when it does -- see `src.awskit.datastores.get_treetracker_results` for the wiring point.

**Needs:** `AWS_RDS_*` in `.env` pointing at the analytics database.

In [ ]:
import json
import os
import sys

import requests
from dotenv import load_dotenv

# Run against a locally started service:  python3 src/main.py pipeline pipeline_api
sys.path.insert(0, os.path.abspath(".."))
load_dotenv(os.path.join("..", ".env"))

BASE_URL = os.getenv("VT_API_BASE_URL", "http://localhost:8000")
TOKEN = os.getenv("API_ENDPOINT_TOKEN")
HEADERS = {"Token": TOKEN}

assert TOKEN, "API_ENDPOINT_TOKEN missing -- copy env_template to .env and fill it in"
print(f"target: {BASE_URL}")

In [ ]:
# Confirm the service is up before sending anything to the route.
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5)
    print(health.status_code, health.json())
except requests.exceptions.ConnectionError:
    print("Service is not running. Start it with:\n"
          "    python3 src/main.py pipeline pipeline_api")

## Request

All filters are optional; omit them to page through everything. `limit` and `offset` are applied by the database, so the response is bounded by the page rather than the table.

In [ ]:
params = {
    "country": null,
    "site": null,
    "session_id": null,
    "limit": 100,
    "offset": 0
}

response = requests.get(f"{BASE_URL}/datascience_results/treetracker_results/",
                        params={k: v for k, v in params.items() if v is not None},
                        headers=HEADERS, timeout=60)

In [ ]:
print(response.status_code)
if response.ok:
    body = response.json()
    print(json.dumps(body, indent=2)[:2000])
else:
    print(response.text[:1000])

## As a DataFrame

In [ ]:
import pandas as pd

if response.ok:
    body = response.json()
    print(f"source     : {body['source']}")
    print(f"n_records  : {body['n_records']}  (total matching the filters)")
    print(f"n_returned : {body['n_returned']}  (this page)")
    df = pd.DataFrame(body["results"])
    display(df.head(20))
else:
    print(response.text[:500])

## Paging

`n_records` is the full count from a `COUNT(*)`; `n_returned` is the size of this page. Walk the result set by advancing `offset`.

In [ ]:
all_rows, offset, page_size = [], 0, 500

while True:
    r = requests.get(f"{BASE_URL}/datascience_results/treetracker_results/",
                     params={"limit": page_size, "offset": offset},
                     headers=HEADERS, timeout=60)
    if not r.ok:
        print(r.status_code, r.text[:300]); break
    body = r.json()
    all_rows.extend(body["results"])
    offset += page_size
    if offset >= body["n_records"] or body["n_returned"] == 0:
        break

print(f"fetched {len(all_rows)} rows")

## Expected response until the upstream table exists

In [ ]:
print(f"status: {response.status_code}")
if response.status_code == 501:
    print("As expected:", response.json()["detail"])